# IMPORT LIBRARY

In [7]:
import pandas as pd
import numpy as np

import torch
import torch._utils
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score

# mlflow
import mlflow
import mlflow.sklearn
import dagshub

# IMPORT DATASET

In [ ]:
import pandas as pd

file_path = "../../data/synthetic_report_dataset.csv" # Pastikan nama/path file-nya benar

# 1. Kita intip dulu file mentahnya untuk melihat pemisahnya
with open(file_path, 'r', encoding='utf-8') as f:
    baris_pertama = f.readline()
    print("Isi baris pertama file mentah:\n👉", baris_pertama)

# 2. Auto-Deteksi Pemisah Kolom
if '|' in baris_pertama:
    pemisah = '|'
else:
    pemisah = ','
print(f"Sistem mendeteksi pemisah kolom: '{pemisah}'\n")

# 3. Load Data
# Menggunakan on_bad_lines='skip' hanya untuk baris yang benar-benar hancur (misal kelebihan koma)
nama_kolom = ['teks_keluhan_awam', 'teks_laporan_teknisi', 'kategori_aset', 'severity', 'root_cause', 'tindakan']
df = pd.read_csv(file_path, sep=pemisah, names=nama_kolom, on_bad_lines='skip')

print(f"Baris sebelum dibersihkan: {df.shape[0]}")

# 4. PEMBERSIHAN & PENYELAMATAN DATA (Pandas Recovery Magic)
kolom_target = ['kategori_aset', 'severity', 'root_cause', 'tindakan']

# a. Deteksi baris patahan (baris yang kepotong enter biasanya kolom targetnya NaN)
baris_patah = df[df['kategori_aset'].isna()].index

# b. Selamatkan teks yang terpotong!
for i in baris_patah:
    # Cek apakah baris di bawahnya ada dan targetnya tidak kosong (berarti itu lanjutannya)
    if i + 1 in df.index and not pd.isna(df.loc[i+1, 'kategori_aset']):
        teks_atas = str(df.loc[i, 'teks_keluhan_awam']).strip()
        teks_bawah = str(df.loc[i+1, 'teks_keluhan_awam']).strip()
        
        # Tempelkan kembali potongan teks ke baris bawahnya
        if teks_atas != 'nan':
            df.loc[i+1, 'teks_keluhan_awam'] = teks_atas + " " + teks_bawah

# c. Setelah teks diselamatkan, baru kita buang baris patahan / NaN tersebut
df = df.dropna(subset=kolom_target)

# d. Buang header CSV asli jika ikut terbaca ke dalam baris data
df = df[df['teks_keluhan_awam'].astype(str).str.contains('teks_keluhan', case=False) == False]

# Reset nomor urut baris agar berurutan kembali
df = df.reset_index(drop=True)

# Pastikan jadi string agar tidak error saat digabung
df['teks_keluhan_awam'] = df['teks_keluhan_awam'].astype(str)
df['teks_laporan_teknisi'] = df['teks_laporan_teknisi'].astype(str)

# 5. Gabungkan Teks (Sebagai Input X)
df['input_teks'] = df['teks_keluhan_awam'] + " " + df['teks_laporan_teknisi']

# 6. MENGAMBIL KOLOM TARGET (Sebagai Kunci Jawaban Y)
Y = df[kolom_target] 

print(f"Total data SIAP DITRAINING: {df.shape[0]}")
print(f"Bentuk Input (X): 1 Kolom Teks Gabungan")
print(f"Bentuk Target (Y): {Y.shape[1]} Kolom Kunci Jawaban")
print("\n--- Intip Isi Target (Y) ---")
display(Y.head(3))

Isi baris pertama file mentah:
👉 AC di ruang meeting lantai 2 netes air terus nih,"Saluran pembuangan air kondensasi tersumbat lumut dan debu, sudah dibersihkan tuntas",HVAC,Sedang,Tersumbat,Pembersihan

Sistem mendeteksi pemisah kolom: ','

Baris sebelum dibersihkan: 2797
Total data SIAP DITRAINING: 2796
Bentuk Input (X): 1 Kolom Teks Gabungan
Bentuk Target (Y): 4 Kolom Kunci Jawaban

--- Intip Isi Target (Y) ---


,kategori_aset,severity,root_cause,tindakan
0,HVAC,Sedang,Tersumbat,Pembersihan
1,Pompa Air,Tinggi,Keausan,Penggantian Part
2,Mesin Produksi,Tinggi,Tersumbat,Pembersihan


# LOAD MODEL INDOBERT

In [9]:
print("\nMemuat otak Deep Learning IndoBERT... (Tunggu sebentar)")
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)


Memuat otak Deep Learning IndoBERT... (Tunggu sebentar)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17036.79it/s]


# FEATURE EXTRACTION

In [10]:
def get_bert_embedding(text):
    # Ubah teks jadi ID Token yang dibatasi 128 kata agar memori tidak penuh
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    # Ambil vektor [CLS] (768 dimensi) yang merangkum makna seluruh kalimat
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

print("Sedang mengekstrak makna kalimat ke dalam 768 dimensi... (Ini mungkin memakan waktu 1-2 menit)")
# Buat matriks X (Embeddings)
X_embeddings = np.vstack(df['input_teks'].apply(get_bert_embedding).values)
y = df[kolom_target]

Sedang mengekstrak makna kalimat ke dalam 768 dimensi... (Ini mungkin memakan waktu 1-2 menit)


# TRAIN-TEST SPLIT

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, y, test_size=0.2, random_state=42)

# MODELING & TUNING

In [12]:
# --- SETUP MLFLOW & DAGSHUB ---
# 1. Inisialisasi DagsHub (Otomatis mengatur tracking URI ke cloud)
dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)

# (Baris set_tracking_uri lokal SUDAH DIHAPUS agar tidak bentrok)

# 2. Set Eksperimen
mlflow.set_experiment("Smart_Ticketing_Baseline_IndoBERT")

# --- MEMULAI TRACKING ---
with mlflow.start_run(run_name="IndoBERT_RF_Hyperparameter_Tuning"):
    
    print("\nMulai melatih model dan mencari parameter terbaik...")
    
    # 1. Mendefinisikan Model dan Grid
    rf_base = RandomForestClassifier(random_state=42)
    multi_target_rf = MultiOutputClassifier(rf_base, n_jobs=1)

    param_grid = {
        'estimator__n_estimators': [100, 200],
        'estimator__max_depth': [None, 10]
    }

    # 2. Proses Latihan (Training)
    grid_search = GridSearchCV(multi_target_rf, param_grid, cv=3, n_jobs=1, verbose=1)
    
    # 🚨 PERBAIKAN: Gunakan .values agar tipe data string Pandas tidak bikin error di Windows!
    grid_search.fit(X_train, y_train.values)

    print(f"\nBest Parameters: {grid_search.best_params_}")
    
    # --- LOGGING KE MLFLOW (DAGSHUB) ---
    # a. Catat Parameter Terbaik
    mlflow.log_params(grid_search.best_params_)
    
    # b. Evaluasi & Catat Metrik
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print("\n=== HASIL EVALUASI FINAL (INDOBERT + RF) ===")
    
    # Hitung & Catat Exact Match
    exact_match_manual = (y_test.values == y_pred).all(axis=1).mean()
    mlflow.log_metric("exact_match_ratio", exact_match_manual)
    print(f"Exact Match Ratio (Benar Semua 4 Kolom): {exact_match_manual * 100:.2f}%")

    print("\n--- Akurasi Individu per Target ---")
    for i, col in enumerate(kolom_target):
        acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
        # Catat metrik per target/kolom ke MLflow
        mlflow.log_metric(f"accuracy_{col}", acc)
        print(f"Akurasi {col:15}: {acc * 100:.2f}%")
        
    # c. Simpan (Log) Model Terbaik
    mlflow.sklearn.log_model(best_model, "best_indobert_rf_model")
    print("\n✅ Run MLflow Selesai! Model dan metrik berhasil dikirim ke DagsHub.")

Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!


Mulai melatih model dan mencari parameter terbaik...
Fitting 3 folds for each of 4 candidates, totalling 12 fits

Best Parameters: {'estimator__max_depth': 10, 'estimator__n_estimators': 200}

=== HASIL EVALUASI FINAL (INDOBERT + RF) ===
Exact Match Ratio (Benar Semua 4 Kolom): 76.25%

--- Akurasi Individu per Target ---
Akurasi kategori_aset  : 99.64%
Akurasi severity       : 76.96%
Akurasi root_cause     : 98.39%


2026/05/09 16:23:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Akurasi tindakan       : 97.50%


2026/05/09 16:23:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run MLflow Selesai! Model dan metrik berhasil dikirim ke DagsHub.
🏃 View run IndoBERT_RF_Hyperparameter_Tuning at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0/runs/abdfe4cfe08f44d496bdb0bfb5ecc328
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0
